## Code to query, visualize, and use output from the HRA Workflow Runner downstream

## Install and import libraries

In [8]:
%pip install duckdb pandas requests hra-jupyter-widgets ipywidgets

import duckdb
import json
from pprint import pprint
import requests
import pandas as pd
import ipywidgets as widgets

from shared import ORGAN_MAPPING

Note: you may need to restart the kernel to use updated packages.


In [9]:
# import Jupyter widgets
from hra_jupyter_widgets import (
    EuiOrganInformation,
    Eui
)

## Global settings

In [10]:
tool_of_interest = 'azimuth'

# get organ from look-up
organ_of_interest = f'http://purl.obolibrary.org/obo/UBERON_{ORGAN_MAPPING["heart"].split(":")[-1]}'

## Fetch and process cell instances

In [11]:
# Combine cell instances and normalize columns for analysis
data_dir = "data/gtex"
query = f"""
SELECT
  split_part(filename, '/', 3) as dataset,
  Organ_ID as organ,
  split_part(filename, '/', 4) as tool,
  column00 as cell,
  clid as cell_id,
  CL_Label as cell_label,
  match_type,
  COALESCE("mapping.score", conf_score, popv_prediction_score / 6, 0) as confidence_score,
FROM read_csv('{ data_dir }/*/*/annotations.csv', union_by_name = true, filename = true, ignore_errors=true, quote='"')
"""

cells = duckdb.sql(query)
cells.write_csv(f'{ data_dir }/cell-instances.csv.gz')
cells.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────────────────┬────────────────┬─────────┬─────────────────────────────────┬────────────┬─────────────────────────────────────┬─────────────────┬────────────────────┐
│            dataset            │     organ      │  tool   │              cell               │  cell_id   │             cell_label              │   match_type    │  confidence_score  │
│            varchar            │    varchar     │ varchar │             varchar             │  varchar   │               varchar               │     varchar     │       double       │
├───────────────────────────────┼────────────────┼─────────┼─────────────────────────────────┼────────────┼─────────────────────────────────────┼─────────────────┼────────────────────┤
│ GTEX-GTEX-12BJ1-5007-SM-H8L6U │ UBERON:0002367 │ popv    │ CST04_TATGCCCGTTCTGAAC-prostate │ CL:0002340 │ luminal cell of prostate epithelium │ skos:exactMatch │                1.0 │
│ GTEX-GTEX-12BJ1-5007-SM-H8L6U │ UBERON:0002367 │ popv    │ CST04_AGATTGCC

In [26]:
df_dvl_llm_4 = cells.to_df()
df_dvl_llm_4

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,dataset,organ,tool,cell,cell_id,cell_label,match_type,confidence_score
0,GTEX-GTEX-12BJ1-5007-SM-H8L6U,UBERON:0002367,popv,CST04_TATGCCCGTTCTGAAC-prostate,CL:0002340,luminal cell of prostate epithelium,skos:exactMatch,1.000000
1,GTEX-GTEX-12BJ1-5007-SM-H8L6U,UBERON:0002367,popv,CST04_AGATTGCCATGGGACA-prostate,CL:0002340,luminal cell of prostate epithelium,skos:exactMatch,0.500000
2,GTEX-GTEX-12BJ1-5007-SM-H8L6U,UBERON:0002367,popv,CST04_TTGACTTGTACACCGC-prostate,CL:0002340,luminal cell of prostate epithelium,skos:exactMatch,1.000000
3,GTEX-GTEX-12BJ1-5007-SM-H8L6U,UBERON:0002367,popv,CST04_CACTCCATCTGGAGCC-prostate,CL:0002340,luminal cell of prostate epithelium,skos:exactMatch,0.833333
4,GTEX-GTEX-12BJ1-5007-SM-H8L6U,UBERON:0002367,popv,CST04_GTGCTTCAGTGAATTG-prostate,CL:0000192,smooth muscle cell,skos:exactMatch,0.833333
...,...,...,...,...,...,...,...,...
331838,GTEX-GTEX-1R9PN-5002-SM-HD2MC,UBERON:0001911,popv,TST01_AGCGTCGGTTGAGTTC-breast,CL:0002326,luminal epithelial cell of mammary gland,skos:exactMatch,1.000000
331839,GTEX-GTEX-1R9PN-5002-SM-HD2MC,UBERON:0001911,popv,TST01_CGGTTAAGTCAAAGAT-breast,CL:0000235,macrophage,skos:exactMatch,0.666667
331840,GTEX-GTEX-1R9PN-5002-SM-HD2MC,UBERON:0001911,popv,TST01_GGGATGAGTACAGTTC-breast,CL:0002326,luminal epithelial cell of mammary gland,skos:exactMatch,0.833333
331841,GTEX-GTEX-1R9PN-5002-SM-HD2MC,UBERON:0001911,popv,TST01_ATAGACCAGTTGAGAT-breast,CL:0002326,luminal epithelial cell of mammary gland,skos:exactMatch,0.833333


In [27]:
df_dvl_llm_4.to_csv('output/dvl-llm-4-gtex-cells.csv', index=False)

In [12]:
# Show only cells annotated with a user-specified tool
cells.filter(f"tool = '{tool_of_interest}'")

┌───────────────────────────────┬──────────────────────┬─────────┬─────────────────────────────┬────────────────────┬────────────┬──────────────────────┬────────────────────┐
│            dataset            │        organ         │  tool   │            cell             │      cell_id       │ cell_label │      match_type      │  confidence_score  │
│            varchar            │       varchar        │ varchar │           varchar           │      varchar       │  varchar   │       varchar        │       double       │
├───────────────────────────────┼──────────────────────┼─────────┼─────────────────────────────┼────────────────────┼────────────┼──────────────────────┼────────────────────┤
│ GTEX-GTEX-13N11-5030-SM-H5JDW │ None                 │ azimuth │ CST03_TCTTTCCTCTGCTTGC-lung │ 0.935159738908882  │ None       │ AT2                  │ 1.0000000000000002 │
│ GTEX-GTEX-13N11-5030-SM-H5JDW │ None                 │ azimuth │ CST03_TCGTACCGTCAGCTAT-lung │ 0.9262700597571508 │ None   

In [14]:

# Cell summary of all cells (counted 3x because run with 3 annotation tools)
cells.aggregate("cell_id, first(cell_label) as cell_label, count(cell) as cell_count").order("cell_count DESC")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────────────────────────────┬───────────────────────────────────────────────────────────┬────────────┐
│                  cell_id                  │                        cell_label                         │ cell_count │
│                  varchar                  │                          varchar                          │   int64    │
├───────────────────────────────────────────┼───────────────────────────────────────────────────────────┼────────────┤
│ CL:0002063                                │ type II pneumocyte                                        │      43255 │
│ CL:0002131                                │ regular ventricular cardiac myocyte                       │      26119 │
│ CL:0000057                                │ fibroblast                                                │      23106 │
│ CL:0002062                                │ type I pneumocyte                                         │      21931 │
│ CL:0000235                                │ ma

In [15]:
# Cell summaries by tool
summaries = cells.aggregate("tool, cell_id, first(cell_label) as cell_label, count(cell) as cell_count").order("tool, cell_count DESC")
df = summaries.to_df()
df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,tool,cell_id,cell_label,cell_count
0,azimuth,CL:0002131,regular ventricular cardiac myocyte,17242
1,azimuth,CL:0002063,type II pneumocyte,14179
2,azimuth,CL:0000057,fibroblast,10803
3,azimuth,CL:0002062,type I pneumocyte,8780
4,azimuth,CL:4028002,alveolar capillary type 1 endothelial cell,3906
...,...,...,...,...
205,popv,ASCTB-TEMP:pneumocyte,pneumocyte,1
206,popv,CL:0000038,erythroid progenitor cell,1
207,popv,CL:0017000,pulmonary ionocyte,1
208,popv,CL:0002394,CD141-positive myeloid dendritic cell,1


## Transform `df` for downstream usage in HRA US#2 UI

In [16]:
# transform df to fit requirements for HRA US#2 at https://apps.humanatlas.io/us2
df['percentage'] = df['cell_count'].apply(lambda c: c/df['cell_count'].sum())

# add column for modality
df['modality'] = 'sc_transcriptomics'

# rename columns as needed
df = df.rename(columns=
  {
    'cell_count':'count'
  }
)

# new column order
new_order = ['tool', 'modality', 'percentage', 'count', 'cell_label','cell_id']

# reassign columns
df_renamed = df[new_order]

df_renamed

,tool,modality,percentage,count,cell_label,cell_id
0,azimuth,sc_transcriptomics,0.051958,17242,regular ventricular cardiac myocyte,CL:0002131
1,azimuth,sc_transcriptomics,0.042728,14179,type II pneumocyte,CL:0002063
2,azimuth,sc_transcriptomics,0.032555,10803,fibroblast,CL:0000057
3,azimuth,sc_transcriptomics,0.026458,8780,type I pneumocyte,CL:0002062
4,azimuth,sc_transcriptomics,0.011771,3906,alveolar capillary type 1 endothelial cell,CL:4028002
...,...,...,...,...,...,...
205,popv,sc_transcriptomics,0.000003,1,pneumocyte,ASCTB-TEMP:pneumocyte
206,popv,sc_transcriptomics,0.000003,1,erythroid progenitor cell,CL:0000038
207,popv,sc_transcriptomics,0.000003,1,pulmonary ionocyte,CL:0017000
208,popv,sc_transcriptomics,0.000003,1,CD141-positive myeloid dendritic cell,CL:0002394


In [17]:
# Export to CSV
df_renamed.to_csv('output/cell_summary.csv', index=False)

## Transform `df` for call to HRA API

In [18]:
# Use HRA API to get predictions: https://apps.humanatlas.io/api/#post-/hra-pop/cell-summary-report
# Convert to CSV string (no index, line breaks handled)
csv_string = df_renamed.to_csv(index=False, lineterminator='\n')

# set URL
url = 'https://apps.humanatlas.io/api/hra-pop/cell-summary-report'

# set headers for the request
headers = {
    'accept': 'application/json',
    'content-type': 'application/json'
}

# set body
body = {
  "csvString": csv_string,
  "organ": organ_of_interest,
  "tool": tool_of_interest
}

# make call and capture response
response = requests.post(url, headers=headers, data=json.dumps(body))

# Check the response
if response.status_code == 200:
    data_dict  = response.json()
    print("Request was successful!")
else:
    print(f"Error: {response.status_code}")

Request was successful!


## Inspect HRA API reponse

In [19]:
# capture response as df, sort to show AS first, then datasets, then extraction sites, and then highest similarity within each
df_response = pd.DataFrame(data_dict ['sources']).sort_values(by=['cell_source_type','similarity'], ascending=[True, False])
df_response

,cell_source,cell_source_type,cell_source_label,cell_source_link,tool,modality,similarity
100,http://purl.org/sig/ont/fma/fma7267,http://purl.org/ccf/AnatomicalStructure,Posteromedial head of posterior papillary musc...,None,azimuth,None,0.266957
107,http://purl.obolibrary.org/obo/UBERON_0002094,http://purl.org/ccf/AnatomicalStructure,interventricular septum,None,azimuth,None,0.264219
112,http://purl.obolibrary.org/obo/UBERON_0002080,http://purl.org/ccf/AnatomicalStructure,heart right ventricle,None,azimuth,None,0.260661
189,http://purl.obolibrary.org/obo/UBERON_0002079,http://purl.org/ccf/AnatomicalStructure,left cardiac atrium,None,azimuth,None,0.184380
206,http://purl.obolibrary.org/obo/UBERON_0002084,http://purl.org/ccf/AnatomicalStructure,heart left ventricle,None,azimuth,None,0.139877
...,...,...,...,...,...,...,...
660,http://purl.org/ccf/1.5/57bab703-98bd-4106-916...,http://purl.org/ccf/SpatialEntity,None,None,azimuth,None,0.012228
687,http://purl.org/ccf/1.5/63b2f5a9-5914-4c45-923...,http://purl.org/ccf/SpatialEntity,None,None,azimuth,None,0.005766
700,http://purl.org/ccf/1.5/502f175c-9d55-4341-b6f...,http://purl.org/ccf/SpatialEntity,None,None,azimuth,None,0.002069
703,http://purl.org/ccf/1.5/6f055461-adb2-41fe-86c...,http://purl.org/ccf/SpatialEntity,None,None,azimuth,None,0.001500


## Visualize extraction sites with EUI 

In [20]:
pprint(data_dict['rui_locations'])

{'@context': 'https://hubmapconsortium.github.io/ccf-ontology/ccf-context.jsonld',
 '@graph': [{'@id': 'https://api.cellxgene.cziscience.com/dp/v1/collections/625f6bf4-2f33-4942-962e-35243d284837#D032_Donor',
             '@type': 'Donor',
             'age': '3',
             'consortium_name': 'NHLBI/LungMap',
             'description': 'Entered 3/24/2023, Allen Wang, NHLBI/LungMap',
             'label': 'Male, Age 3',
             'link': 'https://data-browser.lungmap.net/explore/projects/20037472-ea1d-4ddb-9cd3-56a11a6f0f76',
             'provider_name': 'NHLBI/LungMap',
             'provider_uuid': '0ec3cf4a-4a28-496a-b66e-df34f4cd32e1',
             'samples': {'@id': 'https://api.cellxgene.cziscience.com/dp/v1/collections/625f6bf4-2f33-4942-962e-35243d284837#D032_Donor_TissueBlock1',
                         '@type': 'Sample',
                         'datasets': [{'@id': 'https://api.cellxgene.cziscience.com/dp/v1/collections/625f6bf4-2f33-4942-962e-35243d284837#D032$lung',

In [21]:
# Step 1: Convert the dict to a JSON string
inline_jsonld_str = json.dumps(data_dict['rui_locations'], indent=0)

In [22]:
# Step 2: Wrap it in a list and convert to JSON string again
data_sources_attr = [inline_jsonld_str]

In [23]:
pprint(data_sources_attr)

['{\n'
 '"@context": '
 '"https://hubmapconsortium.github.io/ccf-ontology/ccf-context.jsonld",\n'
 '"@graph": [\n'
 '{\n'
 '"@id": '
 '"https://api.cellxgene.cziscience.com/dp/v1/collections/625f6bf4-2f33-4942-962e-35243d284837#D032_Donor",\n'
 '"samples": {\n'
 '"@id": '
 '"https://api.cellxgene.cziscience.com/dp/v1/collections/625f6bf4-2f33-4942-962e-35243d284837#D032_Donor_TissueBlock1",\n'
 '"@type": "Sample",\n'
 '"donor": '
 '"https://api.cellxgene.cziscience.com/dp/v1/collections/625f6bf4-2f33-4942-962e-35243d284837#D032_Donor",\n'
 '"datasets": [\n'
 '{\n'
 '"@id": '
 '"https://api.cellxgene.cziscience.com/dp/v1/collections/625f6bf4-2f33-4942-962e-35243d284837#D032$lung",\n'
 '"@type": "Dataset",\n'
 '"technology": "OTHER",\n'
 '"thumbnail": "assets/logo.jpg",\n'
 '"link": '
 '"https://data-browser.lungmap.net/explore/projects/20037472-ea1d-4ddb-9cd3-56a11a6f0f76",\n'
 '"description": "Data/Assay Types: OTHER, ",\n'
 '"label": "Registered 3/24/2023, Allen Wang, NHLBI/LungMap"\n

In [24]:
# Convert to JSON-LD string
jsonld_str = json.dumps(data_dict ['rui_locations'], indent=2)

eui_component = Eui(selected_organs = [organ_of_interest], 
                                    data_sources = data_sources_attr
                                    # data_sources = ['https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/v0.12.0/output-data/v0.12.0/atlas-dataset-graph.jsonld']
                                    )
display(eui_component)

Eui(data_sources=['{\n"@context": "https://hubmapconsortium.github.io/ccf-ontology/ccf-context.jsonld",\n"@gra…

In [25]:
target = requests.get('https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/v0.12.0/output-data/v0.12.0/atlas-dataset-graph.jsonld')
pprint(target.text)

('{\n'
 ' "@context": {\n'
 '  "CL": {\n'
 '   "@id": "http://purl.obolibrary.org/obo/CL_",\n'
 '   "@prefix": true\n'
 '  },\n'
 '  "ASCTB-TEMP": {\n'
 '   "@id": "https://purl.org/ccf/ASCTB-TEMP_",\n'
 '   "@prefix": true\n'
 '  },\n'
 '  "ctpop": {\n'
 '   "@id": "https://purl.humanatlas.io/graph/hra-pop#",\n'
 '   "@prefix": true\n'
 '  },\n'
 '  "as_3d_id": {\n'
 '   "@type": "@id"\n'
 '  },\n'
 '  "as_id": {\n'
 '   "@type": "@id"\n'
 '  },\n'
 '  "all_collisions": {\n'
 '   "@id": "ccf:has_collision_summary"\n'
 '  },\n'
 '  "collision_source": {\n'
 '   "@reverse": "ccf:has_collision_summary",\n'
 '   "@type": "@id"\n'
 '  },\n'
 '  "collisions": {\n'
 '   "@id": "ccf:has_collision_item"\n'
 '  },\n'
 '  "corridor_source": {\n'
 '   "@reverse": "ccf:has_corridor",\n'
 '   "@type": "@id"\n'
 '  },\n'
 '  "corridor": {\n'
 '   "@id": "ccf:has_corridor"\n'
 '  },\n'
 '  "aggregated_cell_source": {\n'
 '   "@id": "ccf:cell_source",\n'
 '   "@type": "@id"\n'
 '  },\n'
 '  "summaries